> # ⚠ TUTORIAL — NON-NORMATIVE
>
> **This notebook is a tutorial. It is *not* the specification.**
>
> The normative reference is **`FITA_FORMAT_STANDARD.md` v1.1 (RATIFIED 2026-08-02)**.
> Where this notebook and the standard disagree, **the standard governs** — and where
> this notebook states a measurement, treat it as illustrative until the standard or
> `fita.validate()` confirms it.
>
> Decision **D-7** demoted this notebook explicitly, because it had been functioning as a
> de-facto second specification while repeating two claims that were measured false:
>
> | Claim it used to make | What was measured |
> |---|---|
> | `SPLIT16` costs "~1.5×10⁻⁵ relative error" | The figure described the *encoder function in isolation* and never the file format. Written files lost the flux irrecoverably (max relative error 3.5×10⁶, every pixel altered). **`SPLIT16` is DELETED** (D-2): the writer refuses to emit it and the reader raises. |
> | `FITA_META` is "ObsCore DM v1.1 compliant" | It was not — nine mandatory columns were missing and the per-column UCDs were never written to the file. **This is now genuinely ObsCore DM v1.2** (D-4), with the full mandatory set and `TUCDn` annotation. |
>
> Both are corrected in place below. To check any file against the ratified standard:
>
> ```bash
> fita conform yourfile.fita
> ```

---

# FITA — Flexible Image Transfer Alpha
## Format Reference Notebook

**Version 1.0 · UranoDyne Project · Ignacio A. Cisneros**

---

This notebook is the living reference for the **FITA** multi-layer astrophysical image format.  
Every section is executable; cells run against the `fita` package installed in `C:\Users\astro\fita`.

### Contents

| # | Section | What you learn |
|---|---------|----------------|
| 1 | [Design philosophy](#1.-Design-Philosophy) | Why FITA exists; key constraints |
| 2 | [File structure](#2.-File-Structure) | HDU layout, extension names |
| 3 | [Spec constants](#3.-Spec-Constants-and-Keyword-Registry) | Every keyword in the registry |
| 4 | [FITALayer](#4.-FITALayer) | The layer dataclass: fields, construction |
| 5 | [Flux encoding](#5.-Flux-Encoding) | Normalisation, stretches, SPLIT16 vs FLOAT32 |
| 6 | [Alpha channel](#6.-Alpha-Channel) | How luminance → transparency |
| 7 | [FITACube](#7.-FITACube) | Container, compositing, SED extraction |
| 8 | [Blend modes](#8.-Blend-Modes) | All 14 modes with visual comparison |
| 9 | [New keywords: ZDP, UNCERT, MASK](#9.-New-Keywords:-ZDP,-UNCERT,-MASK) | Stereo depth, uncertainty, quality mask |
| 10 | [FITS backend](#10.-FITS-Backend) | Write/read .fita files |
| 11 | [HDF5 backend](#11.-HDF5-Backend) | Chunked parallel storage |
| 12 | [Zarr backend](#12.-Zarr-Backend) | Cloud-native streaming |
| 13 | [Cross-backend fidelity](#13.-Cross-Backend-Fidelity) | Round-trip comparison |
| 14 | [IVOA provenance](#14.-IVOA-Provenance) | ObsCore DM v1.1 compliance |
| 15 | [Format landscape](#15.-Format-Landscape) | FITA vs HDF5 vs Zarr vs NetCDF vs FITS |
| 16 | [FITR sibling format](#16.-FITR-Sibling-Format) | Radio/interferometry companion |


In [1]:
# ── Environment check ─────────────────────────────────────────────────────────
import sys, importlib

required = {
    'numpy':   'np',
    'matplotlib': 'mpl',
    'astropy': 'astropy',
    'h5py':    'h5py',
    'zarr':    'zarr',
    'fita':    'fita',
}

print(f'Python {sys.version.split()[0]}')
for pkg, alias in required.items():
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', '?')
        status = f'  OK  v{ver}'
    except ImportError:
        status = '  MISSING -- pip install ' + pkg
    print(f'  {pkg:<12} {status}')

Python 3.12.10
  numpy          OK  v2.4.6
  matplotlib     OK  v3.10.9
  astropy        OK  v7.2.0
  h5py           OK  v3.16.0
  zarr           OK  v3.2.1
  fita           OK  v1.0.0


---
## 1. Design Philosophy

FITA was designed to solve a specific problem in multi-band astrophysics:  
**calibrated flux must be preserved exactly while display properties remain fully adjustable.**

### The core invariant

```
layer.flux_data  ←  sacred: this IS the physics (Jy, ct/s, erg/cm²/s/Å …)
layer.alpha_data ←  derived: display-only transparency, never modifies flux
```

### What FITA adds to plain FITS

| Feature | Plain FITS | FITA |
|---------|-----------|------|
| Multi-band as layers | Multiple files or NAXIS3 cube | Single file, per-layer WCS |
| Per-layer alpha mask | Not defined | `ALPHA_*` extension (uint16, 0–65535) |
| Blend modes | Not defined | 14 Photoshop-compatible modes |
| Layer compositing | Not defined | `FITACube.composite()` |
| Stereo depth | Not defined | `FITA_ZDP` keyword |
| Uncertainty map | Convention only | `UNCERT_*` extension |
| Quality bitmask | Convention only | `MASK_*` extension |
| IVOA provenance | Optional HISTORY | Structured `FITA_META` BINTABLE |
| Cloud streaming | No | Zarr backend |
| Parallel I/O | No | HDF5 backend |

### Key design rules

1. **FITS-first** — `.fita` files open in any standard FITS viewer (QFitsView, DS9, FITS Liberator)
2. **No new magic bytes** — SIMPLE = T; fully backward-compatible FITS
3. **Science-safe** — stretch/tone operations only touch the alpha channel; flux_data is read-only by convention
4. **Storage-agnostic** — same data model over FITS, HDF5, or Zarr
5. **IVOA-aligned** — headers use UCDs; provenance follows ObsCore DM v1.1

---
## 2. File Structure

A `.fita` file is a standard multi-extension FITS file (MEF).  
The HDU order is fixed:

```
HDU 0   PRIMARY        Empty data array + global keywords
             FITAVER   = '1.1'
             FITAPACK  = 'FLOAT32'   (SPLIT16 deleted in v1.1, D-2)
             FITANL    = <number of layers>
             FITACW/CH = canvas width / height (optional)
             BUNIT     = flux unit (e.g. 'ct/s')

HDU 1   FITA_LAYERS    BINTABLE — layer registry (one row per layer)
             Columns: LAYER_ID, NAME, EXTNAME_FL, EXTNAME_AL,
                      BLEND_MODE, OPACITY, XOFFSET, YOFFSET,
                      WAVE_CVAL, WAVE_BWID, FLUX_MIN, FLUX_MAX,
                      ALPHA_SRC, VISIBLE, ZDEPTH

HDU 2   FLUX_0001      IMAGE  (float32 or int16 + BSCALE/BZERO)
             Per-layer keywords: FITA_LID, FITA_LNM, FITA_BLD, FITA_OPC,
                                 FITA_XOF, FITA_YOF, FITA_FMN, FITA_FMX,
                                 FITA_WCV, FITA_WBW, FITA_ALS, FITA_ZDP,
                                 FITA_UNC (if uncertainty present),
                                 FITA_MSK (if mask present)
             + standard WCS (CRPIX, CRVAL, CD matrix, CTYPE, CUNIT ...)

HDU 3   ALPHA_0001     IMAGE  BITPIX=16, uint16, range 0–65535
             0     = fully transparent
             65535 = fully opaque

HDU 4   UNCERT_0001    IMAGE  float32  [OPTIONAL]
             1-sigma per-pixel uncertainty in same units as FLUX_0001

HDU 5   MASK_0001      IMAGE  uint8    [OPTIONAL]
             Bitmask: bit0=bad, bit1=saturated, bit2=cosmic ray, bit3=gap

HDU 6   FLUX_0002      (next layer ...)
HDU 7   ALPHA_0002
...
HDU N   FITA_META      BINTABLE  [OPTIONAL]  IVOA ObsCore provenance
```

Extension names follow the templates:
- `FLUX_{id:04d}` → `FLUX_0001`, `FLUX_0042`, ...
- `ALPHA_{id:04d}` → `ALPHA_0001`, ...
- `UNCERT_{id:04d}`, `MASK_{id:04d}` (optional companions)

In [2]:
# Visualise the HDU layout of a freshly created FITA file
import numpy as np
import tempfile
from pathlib import Path
from astropy.io import fits

from fita.layer import FITALayer
from fita.io import write as fits_write, read as fits_read

# Build two synthetic layers
rng = np.random.default_rng(42)

flux_ha   = rng.exponential(scale=50, size=(128, 128)).astype(np.float32)
flux_oiii = rng.exponential(scale=30, size=(128, 128)).astype(np.float32)

l_ha   = FITALayer.from_array(flux_ha,   layer_id=1, name='H-alpha 656nm', wave_cval=656e-9)
l_oiii = FITALayer.from_array(flux_oiii, layer_id=2, name='OIII 501nm',    wave_cval=501e-9)
l_ha.uncert_data  = flux_ha   * 0.05          # 5% photon noise
l_ha.mask_data    = (flux_ha > 200).astype(np.uint8)  # flag saturated pixels
l_ha.zdepth       = 0.3
l_oiii.zdepth     = 0.7

tmp_fita = Path(tempfile.gettempdir()) / 'guide_demo.fita'
fits_write(tmp_fita, [l_ha, l_oiii])

# Inspect the HDU list
with fits.open(str(tmp_fita)) as hdul:
    hdul.info()
    print()
    print('PRIMARY header keywords:')
    for k in ['FITAVER', 'FITAPACK', 'FITANL', 'BUNIT']:
        print(f'  {k:12s} = {hdul[0].header.get(k, "--")!r}')

Filename: C:\Users\astro\AppData\Local\Temp\guide_demo.fita
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      10   ()      
  1  FITA_LAYERS    1 BinTableHDU     43   2R x 15C   [J, 32A, 16A, 16A, 8A, E, D, D, D, D, D, D, 8A, L, E]   
  2  FLUX_0001     1 ImageHDU        22   (128, 128)   float32   
  3  ALPHA_0001    1 ImageHDU        11   (128, 128)   int16   
  4  UNCERT_0001    1 ImageHDU        12   (128, 128)   float32   
  5  MASK_0001     1 ImageHDU        12   (128, 128)   uint8   
  6  FLUX_0002     1 ImageHDU        20   (128, 128)   float32   
  7  ALPHA_0002    1 ImageHDU        11   (128, 128)   int16   

PRIMARY header keywords:
  FITAVER      = '1.0'
  FITAPACK     = 'FLOAT32'
  FITANL       = 2
  BUNIT        = 'ct/s'


In [3]:
# Inspect the FITA_LAYERS registry BINTABLE
from astropy.io import fits
import numpy as np

with fits.open(str(tmp_fita)) as hdul:
    t = hdul['FITA_LAYERS'].data
    cols = t.names
    print(f'FITA_LAYERS has {len(t)} rows, {len(cols)} columns:')
    print()
    for col in cols:
        print(f'  {col:<14}: {list(t[col])}')

FITA_LAYERS has 2 rows, 15 columns:

  LAYER_ID      : [np.int32(1), np.int32(2)]
  NAME          : ['H-alpha 656nm', 'OIII 501nm']
  EXTNAME_FL    : ['FLUX_0001', 'FLUX_0002']
  EXTNAME_AL    : ['ALPHA_0001', 'ALPHA_0002']
  BLEND_MODE    : ['NORMAL', 'NORMAL']
  OPACITY       : [np.float32(1.0), np.float32(1.0)]
  XOFFSET       : [np.float64(0.0), np.float64(0.0)]
  YOFFSET       : [np.float64(0.0), np.float64(0.0)]
  WAVE_CVAL     : [np.float64(6.56e-07), np.float64(5.01e-07)]
  WAVE_BWID     : [np.float64(0.0), np.float64(0.0)]
  FLUX_MIN      : [np.float64(0.21490739285945892), np.float64(0.1247599720954895)]
  FLUX_MAX      : [np.float64(265.78350830078125), np.float64(162.2993927001953)]
  ALPHA_SRC     : ['LUM', 'LUM']
  VISIBLE       : [np.True_, np.True_]
  ZDEPTH        : [np.float32(0.3), np.float32(0.7)]


---
## 3. Spec Constants and Keyword Registry

All keyword strings, blend mode codes, and extension name templates live in `fita.spec`.  
No magic strings anywhere in the codebase — always import from spec.

In [4]:
import fita.spec as spec

print('=== Primary HDU keywords ===')
primary_kw = ['KW_VERSION', 'KW_PACK', 'KW_NLAYERS', 'KW_CANVAS_W', 'KW_CANVAS_H', 'KW_BUNIT']
for name in primary_kw:
    print(f'  {name:<18} = {getattr(spec, name)!r}')

print()
print('=== Layer-level keywords (in each FLUX_* header) ===')
layer_kw = [
    'KW_LAYER_ID', 'KW_LAYER_NAME', 'KW_BLEND_MODE', 'KW_OPACITY',
    'KW_FLUX_MIN', 'KW_FLUX_MAX', 'KW_WAVE_CVAL', 'KW_WAVE_BWID',
    'KW_XOFFSET', 'KW_YOFFSET', 'KW_ALPHA_SRC',
    'KW_DEPTH', 'KW_UNCERT_EXT', 'KW_MASK_EXT',
]
for name in layer_kw:
    print(f'  {name:<20} = {getattr(spec, name)!r}')

print()
print('=== Packing modes ===')
print(f'  PACK_FLOAT32 = {spec.PACK_FLOAT32!r}  (default; lossless 32-bit float)')
print(f'  PACK_SPLIT16 = {spec.PACK_SPLIT16!r}  (lossy; flux + alpha both uint16)')

print()
print('=== Extension name templates ===')
for i in [1, 12, 999]:
    print(f'  layer_id={i:3d} -> flux={spec.flux_extname(i)!r:14s}  alpha={spec.alpha_extname(i)!r}')

=== Primary HDU keywords ===
  KW_VERSION         = 'FITAVER'
  KW_PACK            = 'FITAPACK'
  KW_NLAYERS         = 'FITANL'
  KW_CANVAS_W        = 'FITACW'
  KW_CANVAS_H        = 'FITACH'
  KW_BUNIT           = 'BUNIT'

=== Layer-level keywords (in each FLUX_* header) ===
  KW_LAYER_ID          = 'FITA_LID'
  KW_LAYER_NAME        = 'FITA_LNM'
  KW_BLEND_MODE        = 'FITA_BLD'
  KW_OPACITY           = 'FITA_OPC'
  KW_FLUX_MIN          = 'FITA_FMN'
  KW_FLUX_MAX          = 'FITA_FMX'
  KW_WAVE_CVAL         = 'FITA_WCV'
  KW_WAVE_BWID         = 'FITA_WBW'
  KW_XOFFSET           = 'FITA_XOF'
  KW_YOFFSET           = 'FITA_YOF'
  KW_ALPHA_SRC         = 'FITA_ALS'
  KW_DEPTH             = 'FITA_ZDP'
  KW_UNCERT_EXT        = 'FITA_UNC'
  KW_MASK_EXT          = 'FITA_MSK'

=== Packing modes ===
  PACK_FLOAT32 = 'FLOAT32'  (default; lossless 32-bit float)
  PACK_SPLIT16 = 'SPLIT16'  (lossy; flux + alpha both uint16)

=== Extension name templates ===
  layer_id=  1 -> flux='FLUX_0001'     

In [5]:
print('=== Blend mode codes (all 14) ===')
blend_codes = sorted(spec.BLEND_CODES)
groups = {
    'Arithmetic':  ['NORMAL', 'ADD', 'MULTIPLY', 'SCREEN', 'DIFF'],
    'Contrast':    ['OVERLAY', 'SOFTLGT', 'HARDLGT'],
    'Exposure':    ['CDODGE', 'CBURN'],
    'HSL':         ['LUM', 'COLOR', 'HUE', 'SAT'],
}
for group, codes in groups.items():
    print(f'\n  {group}:')
    for c in codes:
        print(f'    {c}')

=== Blend mode codes (all 14) ===

  Arithmetic:
    NORMAL
    ADD
    MULTIPLY
    SCREEN
    DIFF

  Contrast:
    OVERLAY
    SOFTLGT
    HARDLGT

  Exposure:
    CDODGE
    CBURN

  HSL:
    LUM
    COLOR
    HUE
    SAT


---
## 4. FITALayer

`FITALayer` is a Python dataclass that holds all data and metadata for a single layer.

In [6]:
import dataclasses
from fita.layer import FITALayer
import numpy as np

print('FITALayer fields:')
print()
for f in dataclasses.fields(FITALayer):
    default = f.default if f.default is not dataclasses.MISSING else \
              '(factory)' if f.default_factory is not dataclasses.MISSING else '(required)'
    print(f'  {f.name:<16} {str(f.type):<40} default={default}')

FITALayer fields:

  flux_data        np.ndarray                               default=(required)
  alpha_data       Optional[np.ndarray]                     default=None
  layer_id         int                                      default=1
  name             str                                      default=
  blend_mode       str                                      default=NORMAL
  opacity          float                                    default=1.0
  xoffset          float                                    default=0.0
  yoffset          float                                    default=0.0
  flux_min         Optional[float]                          default=None
  flux_max         Optional[float]                          default=None
  wave_cval        Optional[float]                          default=None
  wave_bwid        Optional[float]                          default=None
  alpha_src        str                                      default=LUM
  visible          bool            

In [7]:
# Two construction paths
import numpy as np
from fita.layer import FITALayer

rng = np.random.default_rng(7)
data = rng.exponential(scale=80, size=(64, 64)).astype(np.float32)

# Path 1: from_array (recommended) — auto-computes flux range + alpha
layer_auto = FITALayer.from_array(
    data,
    layer_id    = 1,
    name        = 'H-alpha',
    stretch_mode= 'asinh',      # display stretch for alpha
    wave_cval   = 656.28e-9,    # metres
    wave_bwid   = 3e-9,
    blend_mode  = 'SCREEN',
    opacity     = 0.9,
)
print('from_array:')
print(f'  flux range  : {layer_auto.flux_min:.2f} – {layer_auto.flux_max:.2f}')
print(f'  alpha dtype : {layer_auto.alpha_data.dtype}  range {layer_auto.alpha_data.min()}–{layer_auto.alpha_data.max()}')
print(f'  shape       : {layer_auto.shape}')
print(f'  alpha_src   : {layer_auto.alpha_src!r}')
print()

# Path 2: manual construction
layer_manual = FITALayer(
    flux_data  = data,
    layer_id   = 2,
    name       = 'Custom',
    flux_min   = 0.0,
    flux_max   = 500.0,
)
print('Manual (no alpha yet):')
print(f'  alpha_data  : {layer_manual.alpha_data}')
print(f'  alpha_float : (all ones — fully opaque)')

# Compute alpha later
layer_manual.recompute_alpha(stretch_mode='log')
print(f'  after recompute_alpha: dtype={layer_manual.alpha_data.dtype}  max={layer_manual.alpha_data.max()}')

from_array:
  flux range  : 0.47 – 439.70
  alpha dtype : uint16  range 0–65535
  shape       : (64, 64)
  alpha_src   : 'LUM'

Manual (no alpha yet):
  alpha_data  : None
  alpha_float : (all ones — fully opaque)
  after recompute_alpha: dtype=uint16  max=65535


In [8]:
# header dict — what gets written to the FITS FLUX_* header
import pprint
layer_auto.zdepth = 0.4
layer_auto.extra_header['OBJECT'] = 'M27'
layer_auto.extra_header['TELESCOP'] = 'SDSS'
pprint.pprint(layer_auto.to_header_dict())

{'FITA_ALS': 'LUM',
 'FITA_BLD': 'SCREEN',
 'FITA_FMN': 0.46539193391799927,
 'FITA_FMX': 439.704345703125,
 'FITA_LID': 1,
 'FITA_LNM': 'H-alpha',
 'FITA_OPC': 0.9,
 'FITA_VIS': True,
 'FITA_WBW': 3e-09,
 'FITA_WCV': 6.5628e-07,
 'FITA_XOF': 0.0,
 'FITA_YOF': 0.0,
 'FITA_ZDP': 0.4,
 'OBJECT': 'M27',
 'TELESCOP': 'SDSS'}


---
## 5. Flux Encoding

The `fita.flux` module handles all display-math operations.  
**The raw flux array is never modified by any of these functions.**

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from fita.flux import normalise, stretch, auto_range, luminance_from_flux, encode_split16

# Simulate a typical astrophysical flux distribution (log-normal, sky background, sources)
rng  = np.random.default_rng(0)
sky  = 5.0
src  = rng.exponential(scale=80, size=(256, 256)).astype(np.float32)
flux = (src + sky).clip(0)

fmin, fmax = auto_range(flux)   # percentile-based robust range
normed      = normalise(flux, fmin, fmax)

stretches   = ['linear', 'sqrt', 'log', 'asinh', 'power']
kw          = {'power': {'power_exp': 0.35}}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes[0, 0].imshow(normed, origin='lower', cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('linear (normalised only)', fontsize=10)
axes[0, 0].axis('off')

for ax, mode in zip(axes.flat[1:], ['sqrt', 'log', 'asinh', 'power']):
    stretched = stretch(normed, mode, **kw.get(mode, {}))
    ax.imshow(stretched, origin='lower', cmap='gray', vmin=0, vmax=1)
    ax.set_title(mode, fontsize=10)
    ax.axis('off')

axes[1, 2].axis('off')
plt.suptitle('Flux stretch functions (same raw data, same flux range, different display stretch)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print(f'auto_range: [{fmin:.2f}, {fmax:.2f}]  (0.5–99.5 percentile of {flux.size:,} pixels)')

auto_range: [5.43, 429.87]  (0.5–99.5 percentile of 65,536 pixels)


C:\Users\astro\AppData\Local\Temp\ipykernel_8952\3954023568.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### SPLIT16 — deleted in v1.1 (decision D-2)

This section used to demonstrate `SPLIT16` packing and quote **"~1.5×10⁻⁵ relative error"**.
That figure was real but described only `encode_split16()` *in isolation*. It never described
what the file format did, and the difference was not small.

**What was measured** (ATOP, 2026-07-29, 10→1000 Jy ramp):

| | Truth | Read back |
|---|---|---|
| Range | `10 .. 1000` | `−32760 .. 32759` |
| Brightest pixel | `1000` | `−1` |
| Pixels altered | — | **4096 of 4096** |
| Max relative error | — | **3.5 × 10⁶** |

Three faults compounded: the encoder's `uint16` output was cast to `int16` without
`BZERO=32768` (wrapping the upper half negative); astropy then discarded the `BSCALE`/`BZERO`
cards because the data was already integer-typed, so the scale factors that define the flux
were **absent from the file**; and the reader fell back to `BSCALE=1, BZERO=0` and returned the
raw wrapped integers as physical flux.

Two further points, which is why D-2 chose deletion over repair:

- The quantisation step is a constant **absolute** quantum, `(FMAX−FMIN)/65535`. It is
  1.5×10⁻⁵ *of the dynamic range*, not of the pixel value — for a faint pixel near `FMIN`
  the relative error approaches 100%.
- `auto_range()` defaults to the 0.5/99.5 percentiles and `normalise()` clips, so a default
  `SPLIT16` write **discarded the brightest and faintest 0.5% of pixels outright** — exactly
  the pixels carrying the science in emission-line and point-source work.

`FLOAT32` is bit-exact (verified: 0 of 4096 pixels altered on a round trip) and is the only
packing a conformant writer may emit. **No archived file was harmed — 0 of 18 used SPLIT16.**

```python
# Both now raise, by design (standard S6.4):
write('out.fita', layers, pack='SPLIT16')   # ValueError
```


---
## 6. Alpha Channel

The alpha channel is FITA's most important innovation: a **physically-derived transparency mask**  
that encodes where each layer contributes signal to the composite.

```
flux_data  →  normalise  →  stretch  →  luminance [0,1]  →  × 65535  →  alpha_data (uint16)
```

- `alpha = 0`     → pixel is transparent (no signal)
- `alpha = 65535` → pixel is fully opaque (maximum signal in this band)

This means **dark sky automatically becomes transparent** — the composite shows whichever layer  
has the brightest signal at any given pixel, without needing manual masking.

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from fita.flux import luminance_from_flux, luminance_to_alpha16, auto_range

rng = np.random.default_rng(1)
# Simulate a nebula: bright core, faint halo, dark background
y, x = np.mgrid[-64:64, -64:64]
r    = np.sqrt(x**2 + y**2).astype(np.float32)
nebula = 500 * np.exp(-r / 15) + 80 * np.exp(-r / 40) + rng.normal(0, 3, r.shape).astype(np.float32)
nebula = nebula.clip(0)

fmin, fmax = auto_range(nebula)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

axes[0].imshow(nebula, origin='lower', cmap='inferno')
axes[0].set_title('Raw flux\n(physical values)', fontsize=9)
axes[0].axis('off')

for ax, mode in zip(axes[1:], ['linear', 'sqrt', 'asinh']):
    lum   = luminance_from_flux(nebula, fmin, fmax, stretch_mode=mode)
    alpha = luminance_to_alpha16(lum)
    ax.imshow(alpha, origin='lower', cmap='gray', vmin=0, vmax=65535)
    ax.set_title(f'Alpha channel\n({mode} stretch)', fontsize=9)
    ax.axis('off')

plt.suptitle('From flux to alpha: dark sky → transparent, bright source → opaque', fontsize=10)
plt.tight_layout()
plt.show()

# Three alpha derivation modes
print('alpha_src modes:')
print('  "LUM"  : alpha derived from flux luminance (default)')
print('  "USER" : alpha supplied by the user (layer.set_user_alpha())')
print('  "NONE" : alpha set to fully opaque (65535 everywhere)')

alpha_src modes:
  "LUM"  : alpha derived from flux luminance (default)
  "USER" : alpha supplied by the user (layer.set_user_alpha())
  "NONE" : alpha set to fully opaque (65535 everywhere)


C:\Users\astro\AppData\Local\Temp\ipykernel_8952\2067107012.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7. FITACube

`FITACube` is the top-level container.  It manages an ordered list of `FITALayer` objects  
and provides compositing, SED extraction, and I/O.

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from fita.cube import FITACube
from fita.layer import FITALayer

rng = np.random.default_rng(3)

# Build a 4-layer cube: Ha, OIII, SII, continuum
def make_nebula(seed, radius, amp, noise, sz=128):
    rng2 = np.random.default_rng(seed)
    y, x = np.mgrid[-sz//2:sz//2, -sz//2:sz//2]
    r    = np.sqrt(x**2 + y**2).astype(np.float32)
    return (amp * np.exp(-r / radius) + rng2.normal(0, noise, r.shape)).astype(np.float32).clip(0)

bands = [
    ('H-alpha 656nm',   656.3e-9, make_nebula(10, 25, 800, 5),  'ADD',    1.0, 0.1),
    ('OIII 501nm',      501.0e-9, make_nebula(11, 32, 400, 4),  'SCREEN', 0.9, 0.4),
    ('SII 672nm',       672.4e-9, make_nebula(12, 20, 300, 3),  'ADD',    0.8, 0.7),
    ('Continuum 550nm', 550.0e-9, make_nebula(13, 40,  50, 8),  'NORMAL', 0.5, 1.0),
]

cube = FITACube(bunit='ct/s')
for i, (name, wave, flux, blend, opacity, zdepth) in enumerate(bands, start=1):
    l = FITALayer.from_array(flux, layer_id=i, name=name, wave_cval=wave,
                              blend_mode=blend, opacity=opacity)
    l.zdepth = zdepth
    cube.layers.append(l)

print(repr(cube))
print()
for l in cube.layers:
    print(f'  [{l.layer_id}] {l.name:<20}  wave={l.wave_cval*1e9:.0f}nm  '
          f'blend={l.blend_mode:8s}  opacity={l.opacity:.1f}  zdepth={l.zdepth}')

FITACube(4 layers, canvas=128x128, bunit='ct/s')

  [1] H-alpha 656nm         wave=656nm  blend=ADD       opacity=1.0  zdepth=0.1
  [2] OIII 501nm            wave=501nm  blend=SCREEN    opacity=0.9  zdepth=0.4
  [3] SII 672nm             wave=672nm  blend=ADD       opacity=0.8  zdepth=0.7
  [4] Continuum 550nm       wave=550nm  blend=NORMAL    opacity=0.5  zdepth=1.0


In [13]:
# Composite and SED
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Individual layers
for i, (ax, layer) in enumerate(zip(axes[:2], cube.layers[:2])):
    ax.imshow(layer.flux_data, origin='lower', cmap='inferno')
    ax.set_title(f'Layer {layer.layer_id}: {layer.name}\nblend={layer.blend_mode}', fontsize=9)
    ax.axis('off')

# Composite
composite_img = cube.composite()
axes[2].imshow(composite_img, origin='lower', cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'FITACube.composite()\n{len(cube.layers)} layers flattened', fontsize=9)
axes[2].axis('off')

plt.tight_layout()
plt.show()

# SED at the brightest pixel
bright_y, bright_x = np.unravel_index(np.argmax(cube.layers[0].flux_data), cube.layers[0].shape)
waves, fluxes = cube.sed(bright_x, bright_y)

fig2, ax2 = plt.subplots(figsize=(7, 3))
ax2.scatter(waves * 1e9, fluxes, color='royalblue', s=80, zorder=3)
ax2.plot(waves * 1e9, fluxes, 'royalblue', lw=1, alpha=0.5)
for w, f, layer in zip(waves, fluxes, cube.layers):
    ax2.annotate(layer.name.split()[0], (w*1e9, f), textcoords='offset points',
                 xytext=(0, 8), fontsize=8, ha='center')
ax2.set_xlabel('Wavelength (nm)')
ax2.set_ylabel('Flux (ct/s)')
ax2.set_title(f'SED at pixel ({bright_x}, {bright_y}) — brightest H-alpha pixel')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\astro\AppData\Local\Temp\ipykernel_8952\4018908875.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\astro\AppData\Local\Temp\ipykernel_8952\4018908875.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8. Blend Modes

FITA implements all 14 Photoshop/GIMP-compatible blend modes.  
Astrophysical note: **ADD** (linear dodge) and **SCREEN** are the most useful for emission-line compositing  
because they accumulate signal without clipping.  **LUM** (Luminosity) is canonical for multi-band  
false-colour: it takes brightness from the science layer while applying colour from a reference image.

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from fita.blend import SCALAR_BLENDS, composite as fita_composite
from fita.spec import BLEND_CODES

rng = np.random.default_rng(5)

# base: faint galaxy halo
y, x  = np.mgrid[-64:64, -64:64].astype(np.float32)
base  = (0.25 * np.exp(-(x**2 + y**2) / 1000)).clip(0, 1)

# blend: bright point sources scattered over it
blend = np.zeros((128, 128), dtype=np.float32)
for _ in range(20):
    cx, cy = rng.integers(10, 118, size=2)
    amp    = rng.uniform(0.3, 1.0)
    blend[cy, cx] = amp
from scipy.ndimage import gaussian_filter
blend = gaussian_filter(blend, sigma=2).astype(np.float32)
blend = (blend / blend.max()).clip(0, 1)

alpha = np.full((128, 128), 0.85, dtype=np.float32)

modes_to_show = ['NORMAL', 'ADD', 'SCREEN', 'MULTIPLY',
                 'OVERLAY', 'SOFTLGT', 'CDODGE', 'CBURN',
                 'DIFF', 'HARDLGT']

fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
axes[0, 0].imshow(base,  cmap='gray', vmin=0, vmax=1, origin='lower')
axes[0, 0].set_title('base', fontsize=9)
axes[0, 0].axis('off')
axes[1, 0].imshow(blend, cmap='gray', vmin=0, vmax=1, origin='lower')
axes[1, 0].set_title('blend layer', fontsize=9)
axes[1, 0].axis('off')

flat_axes = list(axes[0, 1:]) + list(axes[1, 1:])
for ax, mode in zip(flat_axes, modes_to_show):
    result = fita_composite(base, blend, alpha, 1.0, mode)
    ax.imshow(result, cmap='gray', vmin=0, vmax=1, origin='lower')
    ax.set_title(mode, fontsize=9)
    ax.axis('off')

plt.suptitle('Blend modes: top-left=base, left-col-bottom=blend, rest=composited results',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

C:\Users\astro\AppData\Local\Temp\ipykernel_8952\723095135.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 9. New Keywords: ZDP, UNCERT, MASK

Three new data products added in FITA 1.0:

| Keyword / Extension | Type | Purpose |
|---------------------|------|---------|
| `FITA_ZDP` | float 0–1 | Stereo parallax depth; 0 = background, 1 = foreground |
| `UNCERT_*` | float32 array | 1-sigma per-pixel uncertainty (same units as FLUX_*) |
| `MASK_*` | uint8 array | Quality bitmask (bit0=bad, bit1=saturated, bit2=CR, bit3=gap) |

### FITA_ZDP — Phased Stereography

In multi-spectral ISM imaging, different wavebands probe different physical depths:
- Radio 21cm → neutral ISM, deepest penetration
- H-alpha → ionised gas, intermediate depth  
- X-ray → hot plasma, foreground halo

Setting `FITA_ZDP` per layer encodes this physical depth as a number.  
A stereo renderer applies differential x-offset proportional to ZDP to produce parallax.

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from fita.layer import FITALayer

rng = np.random.default_rng(42)

# Simulate three ISM depth planes
def ism_plane(seed, scale, sz=128):
    rng2 = np.random.default_rng(seed)
    img  = rng2.exponential(scale=scale, size=(sz, sz)).astype(np.float32)
    from scipy.ndimage import gaussian_filter
    return gaussian_filter(img, sigma=3)

layers_ism = [
    FITALayer.from_array(ism_plane(1, 100), layer_id=1, name='21cm HI (background)',
                          wave_cval=0.211, blend_mode='ADD'),
    FITALayer.from_array(ism_plane(2,  80), layer_id=2, name='H-alpha (mid)',
                          wave_cval=656e-9, blend_mode='SCREEN'),
    FITALayer.from_array(ism_plane(3,  40), layer_id=3, name='X-ray (foreground)',
                          wave_cval=1e-10,  blend_mode='SCREEN'),
]

# Assign ZDP depths
zdepths = [0.0, 0.5, 1.0]
for l, zd in zip(layers_ism, zdepths):
    l.zdepth = zd

# Compute stereo parallax offsets (for illustration)
baseline_pix = 12   # pixels of parallax for the full depth range

print('Phased stereography — ZDP → parallax offset:')
print(f'{"Layer":<28}  {"ZDP":>5}  {"Parallax offset":>16}  {"Waveband"}')
print('-' * 70)
for l in layers_ism:
    offset = l.zdepth * baseline_pix
    w = l.wave_cval
    ws = f'{w:.3f} m' if w > 1e-3 else (f'{w*1e9:.0f} nm' if w > 1e-7 else f'{w*1e10:.1f} AA')
    print(f'{l.name:<28}  {l.zdepth:>5.1f}  {offset:>+14.1f} px  {ws}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
cmaps = ['Blues_r', 'Reds_r', 'hot']
for ax, l, cmap in zip(axes, layers_ism, cmaps):
    ax.imshow(l.flux_data, origin='lower', cmap=cmap)
    ax.set_title(f'{l.name}\nFITA_ZDP = {l.zdepth}', fontsize=9)
    ax.axis('off')
plt.suptitle('Three ISM depth planes — ZDP encodes physical penetration depth', fontsize=10)
plt.tight_layout()
plt.show()

Phased stereography — ZDP → parallax offset:
Layer                           ZDP   Parallax offset  Waveband
----------------------------------------------------------------------
21cm HI (background)            0.0            +0.0 px  0.211 m
H-alpha (mid)                   0.5            +6.0 px  656 nm
X-ray (foreground)              1.0           +12.0 px  1.0 AA


C:\Users\astro\AppData\Local\Temp\ipykernel_8952\1946405427.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# UNCERT and MASK demonstration
import numpy as np
import matplotlib.pyplot as plt
from fita.layer import FITALayer

rng = np.random.default_rng(99)
sz  = 128

# Realistic science image: galaxy
y, x   = np.mgrid[-sz//2:sz//2, -sz//2:sz//2].astype(np.float32)
r      = np.sqrt(x**2 + y**2)
galaxy = (3000 * np.exp(-r / 12) + 200 * np.exp(-r / 40)).astype(np.float32)
galaxy += rng.normal(0, 15, galaxy.shape).astype(np.float32)   # read noise
galaxy = galaxy.clip(0)

# Uncertainty map: Poisson + read noise
read_noise = 15.0
uncert = np.sqrt(galaxy + read_noise**2).astype(np.float32)

# Quality mask
mask = np.zeros((sz, sz), dtype=np.uint8)
# bad pixels (bit 0)
bad_y, bad_x = rng.integers(0, sz, size=(2, 30))
mask[bad_y, bad_x] |= 0b0001
# saturated pixels near core (bit 1)
mask[r < 6] |= 0b0010
# cosmic rays (bit 2)
cr_y, cr_x = rng.integers(5, sz-5, size=(2, 8))
mask[cr_y, cr_x] |= 0b0100

layer_sci = FITALayer.from_array(galaxy, layer_id=1, name='Galaxy R-band')
layer_sci.uncert_data = uncert
layer_sci.mask_data   = mask

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

axes[0].imshow(galaxy, origin='lower', cmap='inferno', vmin=0, vmax=500)
axes[0].set_title('flux_data\n(FLUX_0001)', fontsize=9)
axes[0].axis('off')

axes[1].imshow(layer_sci.alpha_data, origin='lower', cmap='gray', vmin=0, vmax=65535)
axes[1].set_title('alpha_data\n(ALPHA_0001)', fontsize=9)
axes[1].axis('off')

axes[2].imshow(uncert, origin='lower', cmap='viridis')
axes[2].set_title('uncert_data\n(UNCERT_0001)', fontsize=9)
axes[2].axis('off')

# Visualise mask as RGB: bad=red, saturated=yellow, CR=blue
mask_rgb = np.zeros((sz, sz, 3), dtype=np.float32)
mask_rgb[mask & 0b0001 > 0] = [1, 0, 0]    # bad: red
mask_rgb[mask & 0b0010 > 0] = [1, 1, 0]    # saturated: yellow
mask_rgb[mask & 0b0100 > 0] = [0, 0.5, 1]  # CR: blue
axes[3].imshow(mask_rgb, origin='lower')
axes[3].set_title('mask_data\n(MASK_0001)\nred=bad  yellow=sat  blue=CR', fontsize=8)
axes[3].axis('off')

plt.suptitle('Four FITA data products for one science layer', fontsize=10)
plt.tight_layout()
plt.show()

print(f'SNR at galaxy core (r<6 pix): {galaxy[r<6].mean() / uncert[r<6].mean():.1f}')
print(f'Bad pixels: {(mask & 0b0001).astype(bool).sum()}')
print(f'Saturated : {(mask & 0b0010).astype(bool).sum()}')
print(f'Cosmic rays: {(mask & 0b0100).astype(bool).sum()}')

SNR at galaxy core (r<6 pix): 46.5
Bad pixels: 30
Saturated : 109
Cosmic rays: 8


C:\Users\astro\AppData\Local\Temp\ipykernel_8952\3854899335.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 10. FITS Backend

The default storage backend — `.fita` files are standard FITS MEFs.  
Open natively in **DS9, QFitsView, FITS Liberator, Aladin** without any plugin.

In [17]:
import numpy as np
import tempfile
from pathlib import Path
from fita.layer import FITALayer
from fita.cube import FITACube
from fita.io import write as fits_write, read as fits_read

rng = np.random.default_rng(11)

# Build a 3-layer science cube
def make_layer(lid, name, wave, seed, sz=64):
    rng2 = np.random.default_rng(seed)
    y, x = np.mgrid[-sz//2:sz//2, -sz//2:sz//2].astype(np.float32)
    r    = np.sqrt(x**2 + y**2)
    flux = (200 * np.exp(-r / 10) + rng2.normal(0, 5, r.shape)).astype(np.float32).clip(0)
    l = FITALayer.from_array(flux, layer_id=lid, name=name, wave_cval=wave)
    l.zdepth      = (lid - 1) / 2.0
    l.uncert_data = np.sqrt(flux + 25).astype(np.float32)
    return l

layers_out = [
    make_layer(1, 'Ha',   656e-9, 10),
    make_layer(2, 'OIII', 501e-9, 11),
    make_layer(3, 'SII',  672e-9, 12),
]

# ── Write FLOAT32 ─────────────────────────────────────────────────────────────
p_f32 = Path(tempfile.gettempdir()) / 'guide_f32.fita'
fits_write(p_f32, layers_out, pack='FLOAT32', bunit='ct/s')

# ── Write SPLIT16 ─────────────────────────────────────────────────────────────
p_s16 = Path(tempfile.gettempdir()) / 'guide_s16.fita'
fits_write(p_s16, layers_out, pack='SPLIT16', bunit='ct/s')

# ── Read back ─────────────────────────────────────────────────────────────────
layers_f32 = fits_read(p_f32)
layers_s16 = fits_read(p_s16)

print(f'FLOAT32 file: {p_f32.stat().st_size / 1024:.1f} kB')
print(f'SPLIT16 file: {p_s16.stat().st_size / 1024:.1f} kB')
print()
print(f'Layers read back: {len(layers_f32)} (FLOAT32),  {len(layers_s16)} (SPLIT16)')
print()

for lo, lf, ls in zip(layers_out, layers_f32, layers_s16):
    err_f32 = np.max(np.abs(lo.flux_data - lf.flux_data))
    err_s16 = np.max(np.abs(lo.flux_data - ls.flux_data))
    print(f'  Layer {lo.layer_id} {lo.name:8s}:  '
          f'FLOAT32 err={err_f32:.2e}   SPLIT16 err={err_s16:.2e}  '
          f'zdepth={lf.zdepth}  uncert={lf.uncert_data is not None}')

FLOAT32 file: 163.1 kB
SPLIT16 file: 137.8 kB

Layers read back: 3 (FLOAT32),  3 (SPLIT16)

  Layer 1 Ha      :  FLOAT32 err=0.00e+00   SPLIT16 err=3.28e+04  zdepth=0.0  uncert=True
  Layer 2 OIII    :  FLOAT32 err=0.00e+00   SPLIT16 err=3.28e+04  zdepth=0.5  uncert=True
  Layer 3 SII     :  FLOAT32 err=0.00e+00   SPLIT16 err=3.28e+04  zdepth=1.0  uncert=True


In [18]:
# FITACube high-level save/load
import tempfile
from pathlib import Path
from fita.cube import FITACube

cube_out = FITACube(layers=layers_out, bunit='ct/s',
                    meta={'OBJECT': 'M27', 'TELESCOP': 'SDSS'})

p_cube = Path(tempfile.gettempdir()) / 'guide_cube.fita'
cube_out.save(p_cube)

cube_in = FITACube.load(p_cube)
print(repr(cube_in))

# Composite
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
ax1.imshow(cube_out.composite(), origin='lower', cmap='gray')
ax1.set_title('Saved cube composite')
ax1.axis('off')
ax2.imshow(cube_in.composite(), origin='lower', cmap='gray')
ax2.set_title('Round-trip loaded cube composite')
ax2.axis('off')
plt.tight_layout()
plt.show()

FITACube(3 layers, canvas=64x64, bunit='ct/s')


C:\Users\astro\AppData\Local\Temp\ipykernel_8952\3134656684.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 11. HDF5 Backend

`fita.backends.hdf5` — same FITA data model, HDF5 storage via `h5py`.

**When to use HDF5 instead of FITS:**
- Layers > 2 GB (FITS single-HDU limit)
- Need chunked parallel I/O (LOFAR, SKA-scale)
- Integration with neuroscience tools (FSLeyes, SPM, MATLAB)
- HPC cluster workflows

**Store layout:**
```
/
  .attrs          global FITA attributes
  /layers/
    /0001/
      .attrs      layer metadata
      flux        float32, chunked+gzip4
      alpha       uint16,  chunked+gzip4
      uncert      float32  [optional]
      mask        uint8    [optional]
      wcs_header  JSON string
  /registry       structured array (mirrors FITA_LAYERS)
```

In [19]:
import h5py
import tempfile
from pathlib import Path
from fita.backends.hdf5 import write as h5_write, read as h5_read, info as h5_info

p_h5 = Path(tempfile.gettempdir()) / 'guide.h5'
h5_write(p_h5, layers_out, compress='gzip', compress_opts=4)

print(f'HDF5 file: {p_h5.stat().st_size / 1024:.1f} kB')
print(f'FITS file: {p_f32.stat().st_size / 1024:.1f} kB  (same data, no compression)')
print()

# Inspect internal structure with h5py directly
def print_h5_tree(name, obj):
    indent = '  ' * name.count('/')
    if isinstance(obj, h5py.Dataset):
        print(f'{indent}/{name.split("/")[-1]:20s}  Dataset  {obj.dtype}  {obj.shape}')
    else:
        print(f'{indent}/{name.split("/")[-1]:20s}  Group')

with h5py.File(str(p_h5), 'r') as f:
    print('Root attributes:')
    for k, v in f.attrs.items():
        v_str = v.decode() if isinstance(v, bytes) else v
        print(f'  {k:<18} = {v_str!r}')
    print()
    print('HDF5 tree:')
    f.visititems(print_h5_tree)

HDF5 file: 143.6 kB
FITS file: 163.1 kB  (same data, no compression)

Root attributes:
  BUNIT              = 'ct/s'
  FITANL             = np.int64(3)
  FITAPACK           = 'FLOAT32'
  FITAVER            = '1.0'
  FITA_BACKEND       = 'HDF5'
  FITA_H5VER         = '2.0.0'

HDF5 tree:
/layers                Group
  /0001                  Group
    /alpha                 Dataset  uint16  (64, 64)
    /flux                  Dataset  float32  (64, 64)
    /uncert                Dataset  float32  (64, 64)
    /wcs_header            Dataset  object  ()
  /0002                  Group
    /alpha                 Dataset  uint16  (64, 64)
    /flux                  Dataset  float32  (64, 64)
    /uncert                Dataset  float32  (64, 64)
    /wcs_header            Dataset  object  ()
  /0003                  Group
    /alpha                 Dataset  uint16  (64, 64)
    /flux                  Dataset  float32  (64, 64)
    /uncert                Dataset  float32  (64, 64)
    /wcs_heade

In [20]:
# info() — zero-load summary
summary = h5_info(p_h5)

print(f'version={summary["version"]}  pack={summary["pack"]}  '
      f'nlayers={summary["nlayers"]}  backend={summary["backend"]}')
print()
print(f'{"ID":<4} {"Name":<18} {"Shape":<14} {"Wave(nm)":>10} {"ZDP":>6} {"Uncert":>7} {"Mask":>6}')
print('-' * 68)
for ls in summary['layers']:
    w = ls['wave_cval']
    ws = f'{w*1e9:.0f}' if w and w < 1e-3 else (f'{w:.3f}m' if w else '?')
    print(f'{ls["layer_id"]:<4} {ls["name"]:<18} {str(ls["shape"]):<14} '
          f'{ws:>10} {str(ls["zdepth"]):>6} {str(ls["has_uncert"]):>7} {str(ls["has_mask"]):>6}')

print()
# Read back and verify
layers_h5 = h5_read(p_h5)
for lo, lh in zip(layers_out, layers_h5):
    ok = np.allclose(lo.flux_data, lh.flux_data, atol=1e-5)
    print(f'  Layer {lo.layer_id}: flux match={ok}  zdepth={lh.zdepth}  uncert={lh.uncert_data is not None}')

version=1.0  pack=FLOAT32  nlayers=3  backend=HDF5

ID   Name               Shape            Wave(nm)    ZDP  Uncert   Mask
--------------------------------------------------------------------
1    Ha                 (64, 64)              656    0.0    True  False
2    OIII               (64, 64)              501    0.5    True  False
3    SII                (64, 64)              672    1.0    True  False

  Layer 1: flux match=True  zdepth=0.0  uncert=True
  Layer 2: flux match=True  zdepth=0.5  uncert=True
  Layer 3: flux match=True  zdepth=1.0  uncert=True


---
## 12. Zarr Backend

`fita.backends.zarr` — same data model, Zarr chunk store.  
Zarr stores are **directory trees** locally, or **object-store prefixes** in the cloud.

**When to use Zarr instead of FITS:**
- Browser streaming (IPFS, HTTP range-request)
- S3/GCS/Azure cloud-native access
- Dask parallel computation on cloud arrays
- Very large mosaics split across shards

**Cloud usage pattern:**
```python
# Write to S3
write('s3://my-bucket/m27_cube.zarr', layers,
       storage_options={'key': '...', 'secret': '...'})

# Read anonymously from public bucket
layers = read('s3://my-public-bucket/m27_cube.zarr',
               storage_options={'anon': True})
```

In [21]:
import zarr
import os
import tempfile
from pathlib import Path
from fita.backends.zarr import write as zarr_write, read as zarr_read, info as zarr_info

p_zarr = Path(tempfile.gettempdir()) / 'guide.zarr'
zarr_write(str(p_zarr), layers_out)

# Zarr is a directory; measure total size
total_kb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fnames in os.walk(str(p_zarr))
    for f in fnames
) / 1024

print(f'Zarr store: {total_kb:.1f} kB (directory at {p_zarr.name})')
print(f'FITS file:  {p_f32.stat().st_size / 1024:.1f} kB')
print()

# Inspect the Zarr store structure
root = zarr.open(str(p_zarr), mode='r')
print('Zarr root attributes:')
for k, v in list(root.attrs.items())[:6]:
    print(f'  {k:<18} = {v!r}')
print()
print('Zarr groups and arrays:')
def print_zarr(obj, prefix=''):
    for k in obj:
        item = obj[k]
        if hasattr(item, 'shape'):
            print(f'  {prefix}{k:<20}  Array  {item.dtype}  {item.shape}')
        else:
            print(f'  {prefix}{k}/')
            print_zarr(item, prefix + '  ')
print_zarr(root)

Zarr store: 117.1 kB (directory at guide.zarr)
FITS file:  163.1 kB

Zarr root attributes:
  FITAVER            = '1.0'
  FITAPACK           = 'FLOAT32'
  FITANL             = 3
  BUNIT              = 'ct/s'
  FITA_BACKEND       = 'ZARR'
  ZARR_VERSION       = '3'

Zarr groups and arrays:
  layers/
    0003/
      uncert                Array  float32  (64, 64)
      flux                  Array  float32  (64, 64)
      wcs_header            Array  uint8  (8192,)
      alpha                 Array  uint16  (64, 64)
    0002/
      uncert                Array  float32  (64, 64)
      wcs_header            Array  uint8  (8192,)
      alpha                 Array  uint16  (64, 64)
      flux                  Array  float32  (64, 64)
    0001/
      alpha                 Array  uint16  (64, 64)
      wcs_header            Array  uint8  (8192,)
      flux                  Array  float32  (64, 64)
      uncert                Array  float32  (64, 64)


In [22]:
# info() and round-trip verification
summary_z = zarr_info(str(p_zarr))

print(f'backend={summary_z["backend"]}  zarr_ver={summary_z["zarr_ver"]}  nlayers={summary_z["nlayers"]}')
print()
layers_z = zarr_read(str(p_zarr))
for lo, lz in zip(layers_out, layers_z):
    ok = np.allclose(lo.flux_data, lz.flux_data, atol=1e-5)
    print(f'  Layer {lo.layer_id}: flux match={ok}  zdepth={lz.zdepth}  uncert={lz.uncert_data is not None}')

backend=ZARR  zarr_ver=3  nlayers=3

  Layer 1: flux match=True  zdepth=0.0  uncert=True
  Layer 2: flux match=True  zdepth=0.5  uncert=True
  Layer 3: flux match=True  zdepth=1.0  uncert=True


---
## 13. Cross-Backend Fidelity

All three backends carry the exact same data model.  
Conversion utilities ship with each backend.

In [23]:
import numpy as np
import matplotlib.pyplot as plt
from fita.backends.hdf5 import convert_fits_to_hdf5, read as h5_read
from fita.backends.zarr import convert_fits_to_zarr, read as zarr_read

# Convert from FITS to both backends
p_h5_conv  = convert_fits_to_hdf5(p_f32)
p_zarr_conv = convert_fits_to_zarr(str(p_f32))

layers_from_fits = fits_read(p_f32)
layers_from_h5   = h5_read(p_h5_conv)
layers_from_zarr = zarr_read(p_zarr_conv)

print(f'{"Backend":<10} {"Layers":>7} {"Max flux err":>14} {"zdepth match":>13} {"uncert":>8}')
print('-' * 58)

for name, llist in [('FITS', layers_from_fits), ('HDF5', layers_from_h5), ('Zarr', layers_from_zarr)]:
    max_err    = max(np.max(np.abs(lo.flux_data - l.flux_data))
                    for lo, l in zip(layers_out, llist))
    zdep_match = all(abs(lo.zdepth - l.zdepth) < 1e-6
                     for lo, l in zip(layers_out, llist))
    has_unc    = all(l.uncert_data is not None for l in llist)
    print(f'{name:<10} {len(llist):>7} {max_err:>14.2e} {str(zdep_match):>13} {str(has_unc):>8}')

# Visual cross-backend comparison for layer 1
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, llist) in zip(axes, [('FITS', layers_from_fits), ('HDF5', layers_from_h5), ('Zarr', layers_from_zarr)]):
    ax.imshow(llist[0].flux_data, origin='lower', cmap='inferno')
    ax.set_title(f'{name} backend\nLayer 1: {llist[0].name}', fontsize=9)
    ax.axis('off')
plt.suptitle('Layer 1 flux data from three backends — visually identical', fontsize=10)
plt.tight_layout()
plt.show()

Backend     Layers   Max flux err  zdepth match   uncert
----------------------------------------------------------
FITS             3       0.00e+00          True     True
HDF5             3       0.00e+00          True     True
Zarr             3       0.00e+00          True     True


C:\Users\astro\AppData\Local\Temp\ipykernel_8952\2152393890.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 14. IVOA Provenance

FITA carries an optional `FITA_META` BINTABLE HDU with an
**IVOA ObsCore DM v1.2** provenance record — all 26 mandatory columns, each annotated with its
UCD as a `TUCDn` keyword. This is what makes a `.fita` file discoverable in VO archives and
TAP services.

> **Corrected (D-7).** This notebook previously claimed *"ObsCore DM v1.1 compliant"*. That was
> an overclaim: nine mandatory columns were missing (`obs_publisher_did`, `s_region`, `s_xel1`,
> `s_xel2`, `t_xel`, `em_xel`, `o_ucd`, `pol_states`, `access_estsize`), and the per-column UCDs
> were defined in the source but **never written to the file**. Decision **D-4** took the table to
> full v1.2 and made it reachable from the writer — before v1.1 there was no parameter through
> which to pass provenance, so `FITA_META` was absent from every archived file.

`access_format` is `application/fits`, **not** `application/fits+alpha`: that type is not
registered with IANA and must not be emitted into provenance metadata as though it were (S3).

```python
write('out.fita', layers, provenance={
    'obs_id': 'MWVO-001', 'facility': 'SkyView', 'instrument': 'DSS2',
    'ra': 299.9, 'dec': 22.7,
    'extra': {'obs_publisher_did': 'ivo://mwvo/fita?MWVO-001',
              'obs_collection': 'MWVO'},
})
# -> the file now validates FITA-FULL
```


In [24]:
from fita.ivoa import make_meta_hdu, sed_wavelength_range
from fita.io import write as fits_write
import tempfile
from pathlib import Path
from astropy.io import fits

# Build wavelength coverage from the layer list
em_min, em_max = sed_wavelength_range(layers_out)
print(f'Wavelength coverage: {em_min*1e9:.1f}nm – {em_max*1e9:.1f}nm')

# Build the provenance HDU
meta_hdu = make_meta_hdu(
    obs_id       = 'M27-SDSS-2005',
    obs_title    = 'M27 Dumbbell Nebula Multi-band Cube',
    facility     = 'SDSS',
    instrument   = 'SDSS Camera',
    target       = 'M27',
    ra           = 299.901,
    dec          =  22.721,
    t_min        = 53553.0,   # MJD
    t_max        = 53554.0,
    t_exptime    = 3600.0,
    em_min       = em_min,
    em_max       = em_max,
    calib_level  = 2,
    nlayers      = len(layers_out),
    access_url   = 'https://example.org/fita/m27_demo.fita',
)

# Write FITA file with provenance
p_ivoa = Path(tempfile.gettempdir()) / 'guide_ivoa.fita'
fits_write(p_ivoa, layers_out)

# Append the FITA_META HDU manually (fits_write does not yet auto-attach)
with fits.open(str(p_ivoa), mode='append') as hdul:
    hdul.append(meta_hdu)

# Read it back
with fits.open(str(p_ivoa)) as hdul:
    hdul.info()
    print()
    t = hdul['FITA_META'].data
    print('ObsCore fields in FITA_META:')
    interesting = ['obs_id', 'obs_title', 'facility_name', 'target_name',
                   's_ra', 's_dec', 't_min', 't_exptime',
                   'em_min', 'em_max', 'calib_level', 'fita_nlayers']
    for col in interesting:
        val = t[col][0]
        if col in ('em_min', 'em_max'):
            val = f'{val*1e9:.2f} nm'
        print(f'  {col:<20}: {val}')

Wavelength coverage: 501.0nm – 672.0nm
Filename: C:\Users\astro\AppData\Local\Temp\guide_ivoa.fita
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      10   ()      
  1  FITA_LAYERS    1 BinTableHDU     43   3R x 15C   [J, 32A, 16A, 16A, 8A, E, D, D, D, D, D, D, 8A, L, E]   
  2  FLUX_0001     1 ImageHDU        21   (64, 64)   float32   
  3  ALPHA_0001    1 ImageHDU        11   (64, 64)   int16   
  4  UNCERT_0001    1 ImageHDU        12   (64, 64)   float32   
  5  FLUX_0002     1 ImageHDU        21   (64, 64)   float32   
  6  ALPHA_0002    1 ImageHDU        11   (64, 64)   int16   
  7  UNCERT_0002    1 ImageHDU        12   (64, 64)   float32   
  8  FLUX_0003     1 ImageHDU        21   (64, 64)   float32   
  9  ALPHA_0003    1 ImageHDU        11   (64, 64)   int16   
 10  UNCERT_0003    1 ImageHDU        12   (64, 64)   float32   
 11  FITA_META     1 BinTableHDU     55   1R x 22C   [32A, 64A, 32A, 32A, 32A, 32A, 16A, I, 64A, D, D, D

---
## 15. Format Landscape

Where FITA sits relative to other scientific data formats:

In [25]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Score matrix: rows = formats, cols = capabilities
# Score 0–3: 0=No, 1=Partial, 2=Good, 3=Excellent

formats = ['FITA/FITS', 'FITA/HDF5', 'FITA/Zarr', 'Plain FITS', 'HDF5 (raw)', 'Zarr (raw)', 'NetCDF-4', 'GRIB2']
caps    = [
    'Layer compositing',
    'Alpha channel',
    'Blend modes',
    'Per-layer WCS',
    'Stereo depth',
    'Uncertainty map',
    'Quality mask',
    'IVOA provenance',
    'FITS viewer compat.',
    'Cloud streaming',
    'Parallel chunk I/O',
    'Astro WCS standard',
    'Radio visibility',
    'Spectral cube',
]

scores = np.array([
# comp  alph  blen  wcs   zdp   unc   msk   ivoa  fits  clou  par   awcs  rad   spec
  [3,    3,    3,    3,    3,    3,    3,    3,    3,    0,    1,    3,    0,    2],  # FITA/FITS
  [3,    3,    3,    3,    3,    3,    3,    3,    1,    1,    3,    3,    0,    2],  # FITA/HDF5
  [3,    3,    3,    3,    3,    3,    3,    3,    0,    3,    3,    3,    0,    2],  # FITA/Zarr
  [0,    0,    0,    2,    0,    1,    1,    1,    3,    0,    0,    3,    2,    3],  # Plain FITS
  [0,    0,    0,    0,    0,    2,    2,    0,    0,    1,    3,    0,    1,    2],  # HDF5 (raw)
  [0,    0,    0,    0,    0,    2,    2,    0,    0,    3,    3,    0,    1,    2],  # Zarr (raw)
  [0,    0,    0,    1,    0,    2,    2,    1,    0,    2,    3,    1,    0,    3],  # NetCDF-4
  [0,    0,    0,    0,    0,    0,    1,    1,    0,    2,    2,    0,    0,    1],  # GRIB2
], dtype=float)

fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.cm.RdYlGn
im = ax.imshow(scores, cmap=cmap, vmin=0, vmax=3, aspect='auto')

ax.set_xticks(range(len(caps)))
ax.set_xticklabels(caps, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(formats)))
ax.set_yticklabels(formats, fontsize=9)

for i in range(len(formats)):
    for j in range(len(caps)):
        s = int(scores[i, j])
        labels = {0: '', 1: '~', 2: 'OK', 3: '++'}
        c = 'white' if s < 2 else 'black'
        ax.text(j, i, labels[s], ha='center', va='center', fontsize=7, color=c, fontweight='bold')

# Highlight FITA rows
for i in range(3):
    ax.add_patch(mpatches.FancyBboxPatch(
        (-0.5, i - 0.5), len(caps), 1.0,
        boxstyle='round,pad=0.05', linewidth=2,
        edgecolor='navy', facecolor='none', zorder=5
    ))

plt.colorbar(im, ax=ax, shrink=0.6, label='0=No  1=Partial  2=Good  3=Full')
plt.title('Format capability matrix  (FITA rows outlined in blue)', fontsize=11, pad=12)
plt.tight_layout()
plt.show()

C:\Users\astro\AppData\Local\Temp\ipykernel_8952\1478885513.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
# File size comparison for a typical 3-band 128x128 cube with uncertainty maps
import os

sizes = {
    'FITS FLOAT32':     p_f32.stat().st_size,
    'FITS SPLIT16':     p_s16.stat().st_size,
    'HDF5 gzip-4':      p_h5.stat().st_size,
    'Zarr Blosc/lz4':   sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fnames in os.walk(str(p_zarr))
        for f in fnames
    ),
}

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
labels  = list(sizes.keys())
values  = [v / 1024 for v in sizes.values()]
colors  = ['#4c72b0', '#4c72b0', '#dd8452', '#55a868']
bars    = ax.barh(labels, values, color=colors)
for bar, val in zip(bars, values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} kB', va='center', fontsize=9)
ax.set_xlabel('File size (kB)  — 3 layers, 128×128, with UNCERT')
ax.set_title('FITA backend storage comparison  (same data, different storage)')
plt.tight_layout()
plt.show()

ref = sizes['FITS FLOAT32']
for name, sz in sizes.items():
    print(f'  {name:<20}: {sz/1024:6.1f} kB  ({sz/ref*100:5.1f}% of FITS FLOAT32)')

  FITS FLOAT32        :  163.1 kB  (100.0% of FITS FLOAT32)
  FITS SPLIT16        :  137.8 kB  ( 84.5% of FITS FLOAT32)
  HDF5 gzip-4         :  143.6 kB  ( 88.0% of FITS FLOAT32)
  Zarr Blosc/lz4      :  117.1 kB  ( 71.8% of FITS FLOAT32)


C:\Users\astro\AppData\Local\Temp\ipykernel_8952\102459022.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 16. FITR Sibling Format

**FITR** (Flexible Interferometric Transfer Record) is the radio/interferometry companion to FITA.

| Aspect | FITA | FITR |
|--------|------|------|
| Primary data | Calibrated sky images | Complex visibilities (uv-space) |
| Storage | FITS MEF / HDF5 / Zarr | HDF5 only |
| Pixel unit | Jy/pixel, ct/s, … | Complex64 (re+im, Jy) |
| Alpha channel | Yes — luminance-derived | No (visibility data has no display alpha) |
| Blend modes | 14 modes | N/A |
| Image planes | Primary product | Derived product (after CLEAN) |
| FITS viewer compat. | Full | No (HDF5-native) |
| IVOA provenance | **ObsCore v1.2** (full mandatory set) | ObsCore v1.2 + RadioVis DM |
| Stereo depth (ZDP) | Yes | Yes (on image planes) |
| Link to FITA | — | `/image/` planes are valid FITA layers |

### FITR group layout (summary)

```
/
  .attrs             FITR_VERSION, FITR_ORIGIN, FITR_FREQ_REF, …
  /vis/
    data    complex64  (Nvis, Nspw, Nchan, Npol)
    uvw     float64    (Nvis, 3)   [metres]
    weight  float32    (Nvis, Npol)
    flag    uint8      (Nvis, Nspw, Nchan, Npol)
  /spw/0000/
    freq    float64    (Nchan,)    [Hz]
  /antenna/
    position, name, diameter
  /image/0001/
    flux, alpha, uncert, mask, wcs_header   ← identical to FITA layer!
    .attrs   FITA_LID, FITA_WCV, FITA_ZDP, beam_major, beam_minor, …
  /cal/
    bandpass, gain
  /provenance/
    .attrs   IVOA ObsCore + RadioVis columns
```

> **Status check (non-normative).** The FITA ⇄ FITR bridge is **designed, not built**.
> `FITR_SPEC.md` is a v0.1 DRAFT with no reference implementation, and
> `FITALayer.from_fitr_image()` does not exist. The "identical to a FITA layer" claim above is
> the *intent*; it has never been exercised against a real FITR file.
>
> One real dependency has now been resolved: `FITR_SPEC.md` §8 delegates its display mathematics
> to FITA's `FITA_ADJ` adjustment stack, and until 2026-08-02 **no file had ever contained that
> HDU** — the sibling specs were individually coherent and jointly broken. Decision **D-3**
> implemented `FITA_ADJ`, so that dependency can now be satisfied.

In [27]:
# Demonstrate the FITA/FITR bridge concept:
# A FITR image plane carries the same attrs as a FITA layer.
# Here we simulate writing and reading a minimal FITR /image/ group.

import h5py
import numpy as np
import tempfile
from pathlib import Path

# Simulate a cleaned LOFAR 144 MHz image (as FITR would store it)
rng = np.random.default_rng(77)
y, x = np.mgrid[-64:64, -64:64].astype(np.float32)
r    = np.sqrt(x**2 + y**2)
radio_flux = (5.0 * np.exp(-r / 20) + rng.normal(0, 0.1, r.shape)).astype(np.float32)
radio_flux = radio_flux.clip(0)

fitr_path = Path(tempfile.gettempdir()) / 'demo_minimal.fitr'

with h5py.File(str(fitr_path), 'w') as f:
    # Root FITR attributes
    f.attrs['FITR_VERSION']  = '0.1'
    f.attrs['FITA_VERSION']  = '1.0'
    f.attrs['FITR_ORIGIN']   = 'LOFAR'
    f.attrs['FITR_FREQ_REF'] = 144e6   # 144 MHz
    f.attrs['FITR_RA']       = 299.901
    f.attrs['FITR_DEC']      =  22.721
    f.attrs['FITR_EPOCH']    = '2021-06-15T03:00:00Z'

    # /image/0001 — CLEAN image at 144 MHz
    ig = f.require_group('image/0001')
    # FITA-compatible layer attributes
    ig.attrs['FITA_LID']  = 1
    ig.attrs['FITA_LNM']  = 'LOFAR 144 MHz'
    ig.attrs['FITA_WCV']  = 144e6           # Hz stored as wave_cval placeholder
    ig.attrs['FITA_ZDP']  = 0.0             # background plane
    ig.attrs['BUNIT']     = 'Jy/beam'
    ig.attrs['beam_major']= 6.0             # arcsec
    ig.attrs['beam_minor']= 5.5
    ig.attrs['beam_pa']   = 42.0            # degrees
    ig.create_dataset('flux', data=radio_flux,  compression='gzip')

    # Fake visibility stub
    vis_g = f.require_group('vis')
    nvis  = 1000
    vis_g.create_dataset('data',   data=rng.normal(0, 1, (nvis, 1, 64, 2)).astype(np.complex64))
    vis_g.create_dataset('uvw',    data=rng.normal(0, 1e4, (nvis, 3)).astype(np.float64))
    vis_g.create_dataset('weight', data=np.ones((nvis, 2), dtype=np.float32))

print(f'FITR demo file: {fitr_path.stat().st_size / 1024:.1f} kB')

# Extract the image plane as a FITALayer using the FITA HDF5 reader convention
with h5py.File(str(fitr_path), 'r') as f:
    ig    = f['image/0001']
    flux  = ig['flux'][()].astype(np.float32)
    attrs = dict(ig.attrs)

from fita.layer import FITALayer
layer_from_fitr = FITALayer.from_array(
    flux,
    layer_id  = int(attrs['FITA_LID']),
    name      = attrs['FITA_LNM'].decode() if isinstance(attrs['FITA_LNM'], bytes) else attrs['FITA_LNM'],
    wave_cval = float(attrs.get('FITA_WCV', 0)),
)
layer_from_fitr.zdepth = float(attrs.get('FITA_ZDP', 0))

print()
print('FITALayer extracted from FITR /image/ plane:')
print(f'  name      = {layer_from_fitr.name!r}')
print(f'  wave_cval = {layer_from_fitr.wave_cval:.2e} Hz  (144 MHz radio)')
print(f'  zdepth    = {layer_from_fitr.zdepth}')
print(f'  shape     = {layer_from_fitr.shape}')
print(f'  alpha range: {layer_from_fitr.alpha_data.min()}–{layer_from_fitr.alpha_data.max()}')

FITR demo file: 1101.4 kB

FITALayer extracted from FITR /image/ plane:
  name      = 'LOFAR 144 MHz'
  wave_cval = 1.44e+08 Hz  (144 MHz radio)
  zdepth    = 0.0
  shape     = (128, 128)
  alpha range: 0–65535


---
## Summary

```
┌────────────────────────────────────────────────────────────────────┐
│                     FITA 1.0  --  Quick Reference                  │
├────────────────────────────────────────────────────────────────────┤
│  File suffix       .fita  (standard FITS MEF)                      │
│  MIME type         application/fits+alpha                          │
│  Python package    fita   (pip install -e .)                       │
│  Science engine    uranodyne  (pipeline, calibration, SED)         │
├────────────────────────────────────────────────────────────────────┤
│  Core data model                                                   │
│    FITALayer      dataclass  flux_data + alpha_data + metadata     │
│    FITACube       container  ordered layers + composite() + sed()  │
├────────────────────────────────────────────────────────────────────┤
│  Key keywords (FLUX_* header)                                      │
│    FITA_LID  layer index       FITA_WCV  wavelength (m)            │
│    FITA_BLD  blend mode        FITA_FMN/FMX  flux range            │
│    FITA_OPC  opacity           FITA_ALS  alpha source              │
│    FITA_ZDP  stereo depth      FITA_UNC/MSK  uncert/mask ext name  │
├────────────────────────────────────────────────────────────────────┤
│  Extensions per layer                                              │
│    FLUX_nnnn   float32 or int16+BSCALE (science flux)              │
│    ALPHA_nnnn  uint16  0–65535          (luminance transparency)    │
│    UNCERT_nnnn float32 1-sigma error    (optional)                 │
│    MASK_nnnn   uint8   quality bitmask  (optional)                 │
├────────────────────────────────────────────────────────────────────┤
│  Backends                                                          │
│    fita.io              FITS/MEF  (default, viewer compatible)     │
│    fita.backends.hdf5   HDF5      (chunked parallel I/O)           │
│    fita.backends.zarr   Zarr      (cloud streaming, S3/GCS)        │
├────────────────────────────────────────────────────────────────────┤
│  FITR sibling  .fitr  HDF5-native, complex64 visibilities,        │
│                image planes are FITA-compatible layers             │
└────────────────────────────────────────────────────────────────────┘
```

**GitHub / project root:** `C:\Users\astro\fita`  
**FITR spec:** `C:\Users\astro\fita\FITR_SPEC.md`  
**Specification module:** `fita.spec`  
**IVOA provenance:** `fita.ivoa`